In [1]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
from time import sleep

In [2]:
URL = "https://www.kinoafisha.info/rating/movies/imdb/"

In [4]:
response = requests.get(URL)
response.raise_for_status()
soup = BeautifulSoup(response.text, "html.parser")

In [13]:
text = soup.get_text(separator="\n")
lines = [line.strip() for line in text.split("\n") if line.strip()]

In [6]:
movies = []
i = 0
n = len(lines)

while i < n:
    line = lines[i]

    if re.match(r'^\d\.\d$', line):
        rating = line
        i += 1

        while i < n and not lines[i].strip():
            i += 1
        if i >= n:
            break

        if re.match(r'^\d{1,3}$', lines[i]):
            rank = lines[i]
            i += 1
        else:
            continue

        if i >= n:
            break

        title = lines[i].strip()
        i += 1

        description = "—"
        if i < n:
            next_line = lines[i]
            if (len(next_line) > 60 or
                (not re.search(r'\d{4}', next_line) and
                 not re.match(r'^[\w\s,()-]+$', next_line))):
                description = next_line.strip()
                i += 1

        skip_list = {"Оцените", "Оцени", "Поделиться", "Смотреть", ""}
        while i < n and lines[i].strip() in skip_list:
            i += 1
        if i >= n:
            break

        genres = lines[i].strip()
        i += 1
        if i >= n:
            break

        info = lines[i].strip()
        i += 1

        year_match = re.search(r'\b(\d{4})\b', info)
        year = year_match.group(1) if year_match else "—"

        country = re.sub(r'.*?\b\d{4}\b[\s,;-]*', '', info).strip()
        country = country if country else "—"

        movies.append({
            "rank": rank,
            "title": title,
            "year": year,
            "country": country,
            "rating": rating,
            "genres": genres,
            "description": description
        })

        while i < n and not re.match(r'^\d\.\d$', lines[i]):
            i += 1

    else:
        i += 1

In [12]:
    print(df.head(250).to_string(index=False))
    filename = "kinoafisha_imdb_top250.csv"
    df.to_csv(filename, index=False, encoding="utf-8-sig")
    print(f"\nДанные сохранены в файл: {filename}")

rank                                                                         title year                                      country rating                                                      genres                                                                                                                 description
   1                                                             Побег из Шоушенка 1994                                          США    9.3                                              драма, триллер                                                                                                                           —
   2                                                                 Крестный отец 1972                                          США    9.2                                              драма, триллер                           Культовая криминальная драма Фрэнсиса Форда Копполы возвращается на большие экраны спустя 50 лет!
   3                        

In [11]:
df = pd.DataFrame(movies)
df = df[["rank", "title", "year", "country", "rating", "genres", "description"]]
display(df.head(250))

,rank,title,year,country,rating,genres,description
0,1,Побег из Шоушенка,1994,США,9.3,"драма, триллер",—
1,2,Крестный отец,1972,США,9.2,"драма, триллер",Культовая криминальная драма Фрэнсиса Форда Ко...
2,3,Темный рыцарь,2008,США,9.1,"драма, мистика, триллер, боевик",—
3,4,Крестный отец 2,1974,США,9.0,"триллер, драма",—
4,5,12 разгневанных мужчин,1957,США,9.0,"драма, детектив",—
...,...,...,...,...,...,...,...
245,246,Банды Вассейпура,2012,—,8.2,"боевик, комедия, криминал",—
246,247,Видимость,2015,Индия,8.2,"триллер, драма, криминал",—
247,248,Быть или не быть,1942,США,8.1,"военный, мелодрама, комедия",—
248,249,Гроздья гнева,1940,США,8.1,драма,—
